##Support Classification

In [ ]:
!pip install -q google-cloud-aiplatform google-cloud-storage

import json
import pandas as pd
from google.cloud import storage
from google.cloud import aiplatform

In [ ]:
pip install --upgrade google-genai


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.3/52.3 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 750.9/750.9 kB 28.0 MB/s eta 0:00:00
  Attempting uninstall: google-genai
    Found existing installation: google-genai 1.67.0
    Uninstalling google-genai-1.67.0:
      Successfully uninstalled google-genai-1.67.0


In [ ]:
import os
from google.colab import auth

auth.authenticate_user()
os.environ["GOOGLE_CLOUD_API_KEY"] = "YOUR_API_KEY"

In [ ]:
from google import genai


PROJECT_ID = "gen-lang-client-0150300232"
LOCATION = "us-central1"  # Vertex AI requires a specific region

aiplatform.init(
    project=PROJECT_ID,
    location=LOCATION,
)

client = genai.Client(
    vertexai=True,
    project=PROJECT_ID,
    location=LOCATION
)

response = client.models.generate_content(
    model="gemini-2.0-flash",
    contents="Hello!"
)

print(response.text)

Hello there! How can I help you today?



In [ ]:
import os, time, json, re
import pandas as pd
import openai
from openai import OpenAI
from typing import List, Dict, Any
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from threading import Lock

In [ ]:
def safe_json_load(text: str) -> Dict[str, Any]:
    if not isinstance(text, str):
        return {}
    text = text.strip()

    # direct parse
    try:
        return json.loads(text)
    except Exception:
        pass

    # extract first JSON block
    m = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if m:
        try:
            return json.loads(m.group(0))
        except Exception:
            return {}

    return {}

def chat_to_text(client, model: str, messages: List[Dict[str, str]], max_tokens: int = 300) -> str:
    for _ in range(3):
        try:
            resp = client.chat.completions.create(
                model=model,
                temperature=0,
                max_tokens=max_tokens,
                messages=messages
            )
            return resp.choices[0].message.content.strip()
        except openai.RateLimitError:
            time.sleep(retry_wait_time)
        except Exception:
            time.sleep(retry_wait_time)
    return ""

def chat_to_json(client, model: str, messages: List[Dict[str, str]], max_tokens: int = 300) -> Dict[str, Any]:
    txt = chat_to_text(client, model, messages, max_tokens=max_tokens)
    return safe_json_load(txt)

## Support Classification

In [ ]:
import json, re, time
from typing import Dict, Any, List, Tuple
import pandas as pd

ID_COL = "comment_id"     # column in both CSVs
TEXT_COL = "comment_text"      # column in both CSVs
TAGS_COL = "Tag"      # ONLY exists in the labeled CSV

SUPPORT_LABELS = [
    "Information Support",
    "Emotional Support",
    "Esteem Support",
    "Tangible Assistance",
    "Group Interaction",
]

# ---- Files ----
# Posts you want to label (no tags)
SUPPORT_INPUT_PATH  = "/content/level2_no_stigma_dataset_with_comments.csv - level2_no_stigma_dataset_with_comments.csv"

# Labeled pool used for neighbors (HAS Tags)
SUPPORT_LABELED_POOL_PATH = "/content/Sexual Violence sampled data - 500 random sampling.csv"

# Similarity mapping (computed earlier)
SIM_JSON_PATH = "/content/no_stigma_k5.json"

# Output
SUPPORT_OUTPUT_PATH = "/content/no_stigma_support_predictions.csv"

In [ ]:
# ======================================================
# CELL 4 — chat_to_text / safe_json_load / chat_to_json  ✅ COPY-PASTE
# Robust to: code fences, trailing commas, True/False/None, extra text, empty outputs
# Upgrades:
#  - Extracts the LAST JSON object block (reduces false matches)
#  - Optional required_keys validation (reusable across schemas)
#  - Coerces "present" from "true"/"false" strings to boolean
# ======================================================

from typing import List, Dict, Any, Optional
import json
import re
import time


def chat_to_text(
    client,
    model: str,
    messages: List[Dict[str, str]],
    max_tokens: int = 350,
    max_retries: int = 3,
    sleep_s: float = 1.2,
) -> str:
    """
    Calls chat.completions and returns text.
    Retries on empty content or API exceptions.
    """
    last_err = None
    last_txt = ""

    for attempt in range(1, max_retries + 1):
        try:
            resp = client.chat.completions.create(
                model=model,
                temperature=0,
                max_tokens=max_tokens,
                messages=messages,
            )
            txt = (resp.choices[0].message.content or "").strip()
            last_txt = txt

            # Retry if empty
            if not txt:
                last_err = ValueError(f"Empty model content (attempt {attempt}).")
                time.sleep(sleep_s)
                continue

            return txt

        except Exception as e:
            last_err = e
            time.sleep(sleep_s)

    print("❌ chat_to_text failed:", last_err)
    if last_txt:
        print("   Last text snippet:", last_txt[:220])
    return ""


def safe_json_load(text: str) -> Dict[str, Any]:
    """
    Robust JSON extraction + light repairs.
    Returns {} if unable to parse a JSON object.
    """
    if not isinstance(text, str):
        return {}
    t = text.strip()
    if not t:
        return {}

    # Strip markdown fences if present
    t = re.sub(r"^```json\s*", "", t, flags=re.IGNORECASE).strip()
    t = re.sub(r"^```\s*", "", t).strip()
    t = re.sub(r"\s*```$", "", t).strip()

    # 1) Direct parse
    try:
        obj = json.loads(t)
        return obj if isinstance(obj, dict) else {}
    except Exception:
        pass

    # 2) Extract JSON-looking blocks; choose the LAST one (models put answer last)
    matches = re.findall(r"\{.*\}", t, flags=re.DOTALL)
    if not matches:
        return {}

    blob = matches[-1].strip()

    # Light repairs:
    # - trailing commas
    blob = re.sub(r",\s*}", "}", blob)
    blob = re.sub(r",\s*]", "]", blob)

    # - Python literals to JSON literals
    blob = re.sub(r"\bTrue\b", "true", blob)
    blob = re.sub(r"\bFalse\b", "false", blob)
    blob = re.sub(r"\bNone\b", "null", blob)

    try:
        obj = json.loads(blob)
        return obj if isinstance(obj, dict) else {}
    except Exception:
        return {}


def chat_to_json(
    client,
    model: str,
    messages: List[Dict[str, str]],
    max_tokens: int = 420,
    max_retries: int = 3,
    required_keys: Optional[List[str]] = None,
    sleep_s: float = 1.2,
) -> Dict[str, Any]:
    """
    Calls chat_to_text and parses JSON.
    Retries if JSON parse fails or required keys are missing.

    required_keys default matches your schema:
      ["present", "evidence", "reasoning"]
    """
    if required_keys is None:
        required_keys = ["present", "evidence", "reasoning"]

    last_txt = ""
    last_err = None

    for attempt in range(1, max_retries + 1):
        txt = chat_to_text(
            client, model, messages,
            max_tokens=max_tokens,
            max_retries=1,      # outer loop controls retries
            sleep_s=sleep_s
        )
        last_txt = txt

        parsed = safe_json_load(txt)

        # Retry if parse failed
        if not parsed:
            last_err = ValueError(f"JSON parse failed (attempt {attempt}). Raw: {txt[:220]}")
            time.sleep(sleep_s)
            continue

        # Coerce "present" if model returns it as a string
        if "present" in parsed and isinstance(parsed.get("present"), str):
            val = parsed["present"].strip().lower()
            if val in ("true", "false"):
                parsed["present"] = (val == "true")

        # Some runs might return "explanation" instead of "reasoning"
        # (optional convenience: map explanation -> reasoning)
        if "reasoning" not in parsed and "explanation" in parsed and "reasoning" in required_keys:
            parsed["reasoning"] = parsed.get("explanation", "")

        # Validate keys
        missing = [k for k in required_keys if k not in parsed]
        if missing:
            last_err = ValueError(f"JSON missing keys {missing} (attempt {attempt}). Parsed: {parsed}")
            time.sleep(sleep_s)
            continue

        return parsed

    print("❌ chat_to_json failed:", last_err)
    if last_txt:
        print("   Last raw snippet:", last_txt[:220])
    return {}

In [ ]:
# ==========================================
# CELL — Canonicalize / Validate SUPPORT Output
# ==========================================

def canon_support_output(per_label_results: Dict[str, Dict[str, Any]]) -> Dict[str, Any]:
    """
    Aggregates binary-per-label outputs into a consistent multi-label structure.

    Ensures:
    - label_presence contains all SUPPORT_LABELS
    - labels aligns with label_presence
    - evidence exists only for present labels
    """

    if not isinstance(per_label_results, dict):
        per_label_results = {}

    label_presence = {}
    evidence = {}

    for lab in SUPPORT_LABELS:
        result = per_label_results.get(lab, {})

        present = False
        ev = ""

        if isinstance(result, dict):
            present = bool(result.get("present", False))
            ev_raw = result.get("evidence", "")
            if isinstance(ev_raw, str):
                ev = ev_raw.strip()

        label_presence[lab] = present

        if present and ev:
            evidence[lab] = ev

    # labels derived from label_presence (source of truth)
    labels = [lab for lab, v in label_presence.items() if v]

    # Auto-assign None if no support labels are present
    if not labels:
        labels = ["None"]
        label_presence["None"] = True
    else:
        label_presence["None"] = False

    return {
        "label_presence": label_presence,
        "labels": labels,
        "evidence": evidence
    }

In [ ]:
from typing import Dict, List, Tuple, Any
import os, json, re, random
import pandas as pd

# ======================================================
# SUPPORT PROMPTING (FEWSHOT) — OPTION 2 (Fallback Prototypes) ✅ FIXED
# COPY-PASTE CELL
#
# Assumes df_in and df_pool are SAME ROW ORDER / SAME IDS
# df_in: input texts (no Tag col needed)
# df_pool: same dataset but HAS Tag column for gold labels
# SIM_JSON_PATH maps target_idx -> [[neighbor_idx, score], ...]
#
# REQUIRED VARIABLES to set before running:
#   SUPPORT_INPUT_PATH = "...csv"
#   SUPPORT_LABELED_POOL_PATH = "...csv"
#   SIM_JSON_PATH = "/content/similar_posts_k3_75_3.json"
#   TEXT_COL = "comment_text"
#   TAGS_COL = "Tag"
# ======================================================

# -------------------------------
# 0) LABELS
# -------------------------------
SUPPORT_LABELS: List[str] = [
    "Information Support",
    "Emotional Support",
    "Esteem Support",
    "Tangible Assistance",
    "Group Interaction",
    "None",
]

# -------------------------------
# 1) DEFINITIONS
# -------------------------------
SUPPORT_DEFINITIONS: Dict[str, str] = {
    "Information Support": (
        "Provision of factual information, advice, recommendations, or knowledge intended to help the "
        "recipient understand, evaluate, or respond to their situation."
    ),
    "Emotional Support": (
        "Expressions of empathy, compassion, care, encouragement, or comfort intended to alleviate "
        "emotional distress or provide psychological reassurance."
    ),
    "Esteem Support": (
        "Affirmations that reinforce the recipient’s value, perspective, competence, or worth."
    ),

    "Tangible Assistance": (
        "Support involving direct personal help OR connection to a broader community or support network. "
        "Includes offers of personal assistance (e.g., 'DM me', 'reach out'), willingness to help directly, "
        "or explicitly directing the recipient toward a community/group providing support or belonging."
    ),

    "Group Interaction": (
        "Social or relational engagement in reply to another post, including gratitude/congratulations or sharing personal experience "
        "to relate (e.g., 'this happened to me too'), without primarily offering advice, direct help, or explicitly referencing a broader community."
    ),

    "None": (
        "No social support category applies OR the comment is irrelevant/bot/mod/spam/administrative and provides no support."
    ),
}

In [ ]:
# ======================================================
# ✅ FEWSHOT + PROTOTYPES (SIM-NEIGHBORS) — UPDATED (NO "Seeking Support")
# - Assumes you REMOVED "Seeking Support" from SUPPORT_LABELS
# - Assumes you REMOVED "Network Support" from SUPPORT_LABELS (merged into Tangible Assistance)
# - Adds Tangible hard-negatives to prevent over-trigger on "we're here / not alone"
#
# REQUIRED VARIABLES to set before running:
#   SUPPORT_INPUT_PATH = "...csv"
#   SUPPORT_LABELED_POOL_PATH = "...csv"
#   SIM_JSON_PATH = "similar_posts_....json"
#   TEXT_COL = "comment_text"
#   TAGS_COL = "Tag"
# ======================================================

from typing import Dict, List, Tuple, Any
import os, json, re, random
import pandas as pd

# -------------------------------
# Fewshot controls
# -------------------------------
FEWSHOT_MODE = "snippet"        # "full" | "truncate" | "snippet"
FEWSHOT_TRUNC_CHARS = 900

K_NEIGHBORS = 5
POS_PER_LABEL_DEFAULT = 3
NEG_PER_LABEL_DEFAULT = 2

RANDOM_SEED = 7
random.seed(RANDOM_SEED)

# -------------------------------
# Light sanitization (optional)
# -------------------------------
def _sanitize(text: str) -> str:
    if not text:
        return ""
    s = str(text)
    repl = {
        r"\brape\b": "sexual assault",
        r"\braped\b": "sexually assaulted",
        r"\braping\b": "sexually assaulting",
        r"\bmolest(ed|ation)?\b": "sexual abuse",
        r"\bforced\b": "coerced",
        r"\bpenetrat(e|ed|ion)\b": "sexual act",
    }
    for pat, rep in repl.items():
        s = re.sub(pat, rep, s, flags=re.IGNORECASE)
    return s

# -------------------------------
# Tag parsing (case-insensitive)
# -------------------------------
# NOTE: parse_tags only keeps labels that exist in SUPPORT_LABELS
_CANON_MAP = {lab.lower(): lab for lab in SUPPORT_LABELS}

def parse_tags(tag_cell: Any) -> List[str]:
    if tag_cell is None:
        return []
    if isinstance(tag_cell, list):
        parts = [str(x).strip() for x in tag_cell if str(x).strip()]
    else:
        s = str(tag_cell).strip()
        if not s or s.lower() == "nan":
            return []
        parts = [p.strip() for p in s.replace(";", ",").split(",") if p.strip()]

    out = []
    for p in parts:
        key = p.strip().lower()
        if key in _CANON_MAP:
            out.append(_CANON_MAP[key])

    # dedupe preserve order
    seen = set()
    deduped = []
    for x in out:
        if x not in seen:
            deduped.append(x)
            seen.add(x)
    return deduped

# -------------------------------
# Snippet helpers (3–25 words)
# -------------------------------
def _cap_words(s: str, max_words: int = 25) -> str:
    s = re.sub(r"\s+", " ", str(s or "")).strip()
    words = s.split()
    if len(words) < 3:
        return ""
    if len(words) > max_words:
        s = " ".join(words[:max_words])
    return s.strip()

def _extract_snippet_generic(text: str, max_words: int = 25) -> str:
    t = _sanitize(str(text or "").strip())
    if not t:
        return ""
    sents = re.split(r'(?<=[.!?])\s+', t)
    sents = [x.strip() for x in sents if x.strip()]
    best = sents[0] if sents else t
    return _cap_words(best, max_words=max_words)

def _fewshot_text_for_prompt(text: str) -> str:
    t = _sanitize(str(text or "").strip())
    if FEWSHOT_MODE == "full":
        return t
    if FEWSHOT_MODE == "truncate":
        return t if len(t) <= FEWSHOT_TRUNC_CHARS else (t[:FEWSHOT_TRUNC_CHARS] + "\n...[TRUNCATED]...")
    return _cap_words(t, max_words=60)

# -------------------------------
# Load data
# -------------------------------
assert os.path.exists(SUPPORT_INPUT_PATH), SUPPORT_INPUT_PATH
assert os.path.exists(SUPPORT_LABELED_POOL_PATH), SUPPORT_LABELED_POOL_PATH
assert os.path.exists(SIM_JSON_PATH), SIM_JSON_PATH

df_in = pd.read_csv(SUPPORT_INPUT_PATH).reset_index(drop=True)
df_pool = pd.read_csv(SUPPORT_LABELED_POOL_PATH).reset_index(drop=True)

with open(SIM_JSON_PATH, "r") as f:
    SIM_MAP: Dict[str, List[List[Any]]] = json.load(f)

print("✅ df_in:", df_in.shape, "| df_pool:", df_pool.shape, "| SIM keys:", len(SIM_MAP))

if TEXT_COL not in df_in.columns:
    raise ValueError(f"TEXT_COL '{TEXT_COL}' not found in df_in. Found: {list(df_in.columns)}")
if TEXT_COL not in df_pool.columns:
    raise ValueError(f"TEXT_COL '{TEXT_COL}' not found in df_pool. Found: {list(df_pool.columns)}")
if TAGS_COL not in df_pool.columns:
    raise ValueError(f"TAGS_COL '{TAGS_COL}' not found in df_pool (needed for fewshots). Found: {list(df_pool.columns)}")

# -------------------------------
# Build global prototypes (indices) per label from df_pool tags
# -------------------------------
pool_tags = df_pool[TAGS_COL].apply(parse_tags)

LABEL_TO_POS_IDXS: Dict[str, List[int]] = {lab: [] for lab in SUPPORT_LABELS}
LABEL_TO_NEG_IDXS: Dict[str, List[int]] = {lab: [] for lab in SUPPORT_LABELS}

for idx, labs in enumerate(pool_tags.tolist()):
    labs_set = set(labs)
    for lab in SUPPORT_LABELS:
        if lab in labs_set:
            LABEL_TO_POS_IDXS[lab].append(idx)
        else:
            LABEL_TO_NEG_IDXS[lab].append(idx)

# ======================================================
# ✅ Tangible Assistance precision booster: "hard negatives"
# Looks like "we're here / not alone" but NO DM/help offer
# (Useful after merging Network -> Tangible)
# ======================================================
_TANG_HARD_NEG_RE = re.compile(
    r"\bwe('?re)?\b|\bus\b|\bour\b|\byou('?re)? not alone\b|\bnot alone\b|\bhere for you\b",
    flags=re.IGNORECASE
)
_TANG_POS_CUE_RE = re.compile(
    r"\bdm\b|\bmessage me\b|\bpm me\b|\breach out\b|\bcontact me\b|\bi can help\b|\bi'll help\b|\bhappy to help\b|\blet me know\b|\bcall me\b",
    flags=re.IGNORECASE
)

TANGIBLE_HARD_NEG_IDXS: List[int] = []
for i in LABEL_TO_NEG_IDXS.get("Tangible Assistance", []):
    txt = "" if pd.isna(df_pool.loc[i, TEXT_COL]) else str(df_pool.loc[i, TEXT_COL])
    if _TANG_HARD_NEG_RE.search(txt or "") and not _TANG_POS_CUE_RE.search(txt or ""):
        TANGIBLE_HARD_NEG_IDXS.append(i)

print("✅ Tangible hard negatives:", len(TANGIBLE_HARD_NEG_IDXS))

# -------------------------------
# Neighbor fetch
# -------------------------------
def get_neighbor_records(target_idx: int, k: int = K_NEIGHBORS) -> List[Dict[str, Any]]:
    raw = (SIM_MAP.get(str(target_idx), []) or [])[:k]
    out = []
    for pair in raw:
        if not isinstance(pair, (list, tuple)) or len(pair) < 1:
            continue
        nb_idx = int(pair[0])
        if nb_idx == target_idx:
            continue
        score = float(pair[1]) if len(pair) > 1 else 1.0
        if nb_idx < 0 or nb_idx >= len(df_pool):
            continue
        nb_text = "" if pd.isna(df_pool.loc[nb_idx, TEXT_COL]) else str(df_pool.loc[nb_idx, TEXT_COL])
        nb_tags = parse_tags(df_pool.loc[nb_idx, TAGS_COL])
        out.append({"idx": nb_idx, "score": score, "text": nb_text, "tags": nb_tags})
    return out

# ======================================================
# ✅ Fewshot picker (neighbors -> fallback prototypes)
# - Tangible uses HARD NEGATIVES first for better precision
# ======================================================
def build_labelwise_fewshots_for_target(target_idx: int, lab: str) -> List[Tuple[str, str]]:
    neighbors = get_neighbor_records(target_idx, k=K_NEIGHBORS)

    pos = [n for n in neighbors if lab in set(n["tags"])]
    neg = [n for n in neighbors if lab not in set(n["tags"])]

    want_pos, want_neg = POS_PER_LABEL_DEFAULT, NEG_PER_LABEL_DEFAULT

    picked: List[Dict[str, Any]] = []

    # 1) positives from neighbors
    picked.extend(pos[:want_pos])

    # 2) fallback positives
    cur_pos = sum(1 for x in picked if lab in set(x.get("tags", [])))
    if cur_pos < want_pos:
        need = want_pos - cur_pos
        candidates = [i for i in LABEL_TO_POS_IDXS.get(lab, []) if i != target_idx]
        random.shuffle(candidates)
        for i in candidates:
            if need <= 0:
                break
            txt = "" if pd.isna(df_pool.loc[i, TEXT_COL]) else str(df_pool.loc[i, TEXT_COL])
            tags = parse_tags(df_pool.loc[i, TAGS_COL])
            picked.append({"idx": i, "score": None, "text": txt, "tags": tags})
            need -= 1

    # 3) negatives from neighbors
    picked.extend(neg[:want_neg])

    # 4) fallback negatives
    have_neg = sum(1 for x in picked if lab not in set(x.get("tags", [])))
    if have_neg < want_neg:
        need = want_neg - have_neg

        if lab == "Tangible Assistance":
            candidates = [i for i in TANGIBLE_HARD_NEG_IDXS if i != target_idx]
            random.shuffle(candidates)
            if len(candidates) < need:
                extra = [i for i in LABEL_TO_NEG_IDXS.get(lab, []) if i != target_idx and i not in set(candidates)]
                random.shuffle(extra)
                candidates = candidates + extra
        else:
            candidates = [i for i in LABEL_TO_NEG_IDXS.get(lab, []) if i != target_idx]
            random.shuffle(candidates)

        for i in candidates:
            if need <= 0:
                break
            txt = "" if pd.isna(df_pool.loc[i, TEXT_COL]) else str(df_pool.loc[i, TEXT_COL])
            tags = parse_tags(df_pool.loc[i, TAGS_COL])
            picked.append({"idx": i, "score": None, "text": txt, "tags": tags})
            need -= 1

    # cap
    max_examples = want_pos + want_neg
    picked = picked[:max_examples]

    # build (text, answer_json)
    fewshots: List[Tuple[str, str]] = []
    for ex in picked:
        ex_text = ex["text"]
        ex_tags = ex["tags"]
        is_pos = (lab in set(ex_tags))

        ev = _extract_snippet_generic(ex_text, max_words=25) if is_pos else ""

        ans = {
            "present": bool(is_pos),
            "evidence": ev if is_pos else "",
            "reasoning": (
                f"Gold pool example (idx={ex['idx']}"
                + (f", sim={ex['score']:.3f}" if isinstance(ex.get('score'), float) else "")
                + f"). Label '{lab}' present={'true' if is_pos else 'false'}."
            )
        }

        fewshots.append((ex_text, json.dumps(ans, ensure_ascii=False)))

    return fewshots

✅ df_in: (1175, 11) | df_pool: (500, 10) | SIM keys: 1175
✅ Tangible hard negatives: 142


In [ ]:
# -------------------------------
# Prompt builder (unchanged, but now fewshots are VALID)
# -------------------------------
def build_support_system_prompt(lab: str, target_idx: int) -> str:
    definition = SUPPORT_DEFINITIONS[lab].strip()

    system_prompt = f"""
You are a JSON-only annotator specializing in SOCIAL SUPPORT classification.
You will evaluate ONE support category at a time.

GLOBAL RULES:
- Use ONLY the comment/reply text. Do NOT infer intent.
- Decide strictly for the label under test: "{lab}".
- "present" is true ONLY if there is explicit evidence matching the label definition and decision boundaries.
- The "evidence" MUST be a verbatim snippet from the comment proving the label (3–25 words).
- If evidence is weak/ambiguous, set "present": false.
- If you cannot quote a 3–25 word verbatim snippet that clearly proves the label, set "present": false.
- Return ONLY valid JSON.

PRIORITY DISAMBIGUATION:
- If the text is primarily advice/steps/resources → prefer Information Support over Group/Network.
- If the text is primarily comfort/empathy → prefer Emotional Support over Group/Network.
- If the text offers direct personal help/contact (DM me, reach out) → Tangible Assistance.
- Group Interaction requires personal experience/relating OR thanks/congrats, and is NOT advice-focused.

LABEL UNDER TEST:
{lab}: {definition}

LABEL-SPECIFIC BOUNDARY RULES:
""".strip()

    if lab == "Information Support":
        system_prompt += "\n" + """
TRUE when primary purpose is to inform, instruct, or guide (advice, recommendations, resources, steps).
FALSE when empathy-only (Emotional Support) OR direct offer to do a task (Tangible Assistance).
""".strip()
    elif lab == "Emotional Support":
        system_prompt += "\n" + """
    TRUE when the primary purpose is comfort/empathy/reassurance (emotional soothing).

    REQUIRES at least one explicit emotional cue (must be quotable):
    - "I'm sorry", "so sorry", "that sounds (awful/hard)", "sending love", "hugs"
    - "you are not alone", "I'm here for you", "take care", "I hear you", "I see you"
    - reassurance like "it's okay to feel...", "you'll get through this", "be gentle with yourself"

    FALSE (precision guardrails):
    - If the comment is primarily steps/resources/what to do → Information Support (set Emotional=false),
      even if it contains a short empathy phrase like "sorry" at the start.
    - If the comment is primarily worth/competence validation ("you're strong/brave", "not your fault", "you did the right thing")
      → Esteem Support (Emotional can be false unless there is substantial comfort beyond validation).
    - If it’s mainly personal experience sharing to relate → Group Interaction.
    - If it’s offering direct help/contact (DM me / reach out) → Tangible Assistance.

    Decision rule:
    - If emotional wording is only 1 short line and the rest is advice/resources, set present=false.
    Evidence MUST quote the emotional cue (3–25 words).
    """.strip()
    elif lab == "Esteem Support":
        system_prompt += "\n" + """
TRUE when affirming worth/competence/perspective (compliments, validation, "you are strong/brave", "not your fault").
FALSE when general comfort without affirming worth (Emotional Support).
""".strip()
    elif lab == "Tangible Assistance":
        system_prompt += "\n" + r"""
    TRUE when the comment provides ASSISTANCE beyond emotion/advice, in either of these ways:

    IMPORTANT (STRICT): Tangible Assistance requires QUOTABLE EVIDENCE that contains an EXPLICIT TRIGGER token.
    If the quoted evidence does NOT contain a trigger token from the lists below, set present=false.

    TRIGGER TOKENS (must appear in evidence quote):

    (A) Direct personal assistance / availability (one-to-one help) triggers:
    - "dm me", "dm", "message me", "pm me", "pm"
    - "reach out", "reach out to me", "contact me"
    - "i can help", "i'll help", "i am happy to help", "happy to help"
    - "let me know if you want to talk", "here if you want to talk", "talk privately"
    - "i can connect you", "i can set you up", "i can walk you through", "walk you through"

    (B) Community / support network / service triggers:
    - "this subreddit", "this sub", "this community", "people here", "others here", "many of us"
    - "support group", "survivor group"
    - "advocacy center", "rape crisis center", "crisis center", "empowerment center"
    - "hotline", "helpline" (ONLY if framed as a support service)

    (A) Direct personal assistance / availability (one-to-one help)
    Set present=true ONLY if there is an explicit personal offer of time/access/action
    AND your evidence quote contains at least one (A) trigger token.
    Evidence MUST quote the offer cue containing the trigger (3–25 words).

    (B) Community / network assistance (merged Network→Tangible)
    Set present=true ONLY if the comment explicitly connects the recipient to a SUPPORT NETWORK / COMMUNITY
    as the source of help AND your evidence quote contains at least one (B) trigger token.
    Evidence MUST quote the anchor/service phrase containing the trigger (3–25 words).

    CRITICAL PRECISION GUARDS (these are NOT Tangible unless a trigger token is present in evidence):
    1) Advice-to-professionals/resources without a personal offer/network anchor:
      - "call the police", "go to the hospital", "see a therapist", "talk to your doctor",
        "report it", "get tested"
      → ALWAYS Information Support unless the text ALSO includes:
        (i) a direct offer trigger (A), OR
        (ii) an explicit support-service/community trigger (B) like "rape crisis center", "support group", "hotline".

    2) Generic reassurance with no offer and no community/service anchor:
      - "we're here for you", "you're not alone", "sending love/hugs"
      → Emotional Support / Esteem Support unless the SAME sentence includes a (B) trigger token
        (e.g., "we here on this sub are here for you", "this community is here for you").

    Special rule for “we’re here for you” (STRICT):
    - TRUE only when the SAME sentence contains a community trigger token (B).
    - Otherwise FALSE.

    Decision rule (strict):
    Tangible Assistance = TRUE only if you can quote either:
    (1) a direct offer/availability cue containing an (A) trigger token, OR
    (2) an explicit support network/community/service anchor containing a (B) trigger token.
    If you cannot quote one WITH a trigger token, set present=false.

    FINAL CHECK:
    If you cannot point to a trigger token inside your quoted evidence, you MUST output present=false.
    """.strip()

    elif lab == "Group Interaction":
        system_prompt += "\n" + """
    TRUE only when the comment is primarily RELATIONAL / SOCIAL engagement in reply to another post,
    and includes at least ONE explicit Group marker.

    Group markers (must be explicit and quotable):
    (A) Relating / personal experience to connect:
    - "this happened to me too", "same here", "I've been there"
    - "when I...", "in my experience...", "I went through..."
    (B) Social acknowledgement / conversational interaction:
    - "thank you for sharing", "thanks for sharing"
    - "congrats", "congratulations", "I'm happy for you"
    - "I appreciate you sharing", "welcome", greetings that are the main point

    ❌ FALSE (main precision guardrails):
    1) If the PRIMARY purpose is advice/steps/resources → Information Support (set Group=false),
      even if it contains personal experience. (Example: "In my experience, you should report it...")
    2) If the PRIMARY purpose is comfort/empathy with no clear (A) or (B) → Emotional Support (Group=false).
    3) If the PRIMARY purpose is validating worth/blame/strength ("not your fault", "you're strong") → Esteem Support (Group=false).
    4) If offering direct help/contact ("DM me", "reach out") → Tangible Assistance (Group=false).
    5) If explicitly pointing to a broader community/group ("this subreddit", "support group", "we as a community") → Tangible Assistance (Group=false).

    Decision rule:
    - ONLY mark Group=true if the relational marker (A/B) is central, not a side sentence.
    - Evidence MUST quote the Group marker phrase (A/B). If you cannot quote it, set present=false.
    """.strip()
    elif lab == "None":
        system_prompt += "\n" + """
TRUE when no support category applies OR comment is irrelevant/bot/mod/spam/admin with no support content.
FALSE when any real support (information/emotional/esteem/network/tangible/seeking/group) is present.
""".strip()

    fewshots = build_labelwise_fewshots_for_target(target_idx, lab)

    system_prompt += f"""

HOW TO USE THE EXAMPLES BELOW:
- The following are gold-labeled examples similar to the one you will evaluate.
- Each example shows how the label "{lab}" can be present (true) or absent (false).
"""

    system_prompt += "\n\nFEW-SHOT EXAMPLES:\n"
    for i, (ex_text, ans_json) in enumerate(fewshots, start=1):
        shown = _fewshot_text_for_prompt(ex_text)
        system_prompt += f"""
EXAMPLE {i}
NEIGHBOR TEXT ({FEWSHOT_MODE}):
{shown}

ANSWER JSON:
{ans_json}
""".rstrip()

    system_prompt += f"""

NOW LABEL THIS TEXT for the "{lab}" category ONLY.

Return ONLY valid JSON in exactly this format:
{{
  "reasoning": "1–3 sentences. State why {lab} is present/absent based on definition + boundaries.",
  "present": true or false,
  "evidence": "verbatim snippet if true (3–25 words), else empty string"
}}
""".strip()

    return system_prompt

def build_final_prompt_for_debug(system_prompt: str, target_text: str) -> str:
    target_text = "" if target_text is None else str(target_text)
    return (
        f"{system_prompt}\n\n"
        f"TEXT:\n{target_text}\n\n"
        "FINAL INSTRUCTIONS:\n"
        "Output ONLY a single JSON object (no markdown, no extra text).\n"
        "Use lowercase true/false.\n"
        "Include exactly these keys: reasoning, present, evidence.\n"
    )

# -------------------------------
# DEBUG: print one Seeking prompt
# -------------------------------
target_idx_debug = 0
lab_debug = "Tangible Assistance"
txt = "" if pd.isna(df_in.loc[target_idx_debug, TEXT_COL]) else str(df_in.loc[target_idx_debug, TEXT_COL])
sys_prompt = build_support_system_prompt(lab_debug, target_idx_debug)
final_prompt = build_final_prompt_for_debug(sys_prompt, txt)

print("\n" + "="*90)
print(f"DEBUG PROMPT (target_idx={target_idx_debug}, label={lab_debug})")
print("="*90)
print(final_prompt)
print("="*90)


DEBUG PROMPT (target_idx=0, label=Tangible Assistance)
You are a JSON-only annotator specializing in SOCIAL SUPPORT classification.
You will evaluate ONE support category at a time.

GLOBAL RULES:
- Use ONLY the comment/reply text. Do NOT infer intent.
- Decide strictly for the label under test: "Tangible Assistance".
- "present" is true ONLY if there is explicit evidence matching the label definition and decision boundaries.
- The "evidence" MUST be a verbatim snippet from the comment proving the label (3–25 words).
- If evidence is weak/ambiguous, set "present": false.
- If you cannot quote a 3–25 word verbatim snippet that clearly proves the label, set "present": false.
- Return ONLY valid JSON.

PRIORITY DISAMBIGUATION:
- If the text is primarily advice/steps/resources → prefer Information Support over Group/Network.
- If the text is primarily comfort/empathy → prefer Emotional Support over Group/Network.
- If the text offers direct personal help/contact (DM me, reach out) → Tangi

In [ ]:
target_idx_debug = 0
print("Neighbors stored in SIM_MAP for target:", len(SIM_MAP.get(str(target_idx_debug), [])))
print("First few neighbors:", SIM_MAP.get(str(target_idx_debug), [])[:10])

Neighbors stored in SIM_MAP for target: 5
First few neighbors: [[216, 0.6463422775268555], [392, 0.6294177174568176], [309, 0.6188831329345703], [454, 0.6136670708656311], [364, 0.6087182760238647]]


In [ ]:
from functools import lru_cache

@lru_cache(maxsize=500 * 8)  # enough for 500 idx × ~8 labels
def cached_system_prompt(lab: str, target_idx: int) -> str:
    return build_support_system_prompt(lab=lab, target_idx=int(target_idx))

In [ ]:
# ======================================================
# ✅ SUPPORT Gemini Runner — UPDATED FOR OPTION 2 (Fallback Prototypes)
# COPY-PASTE CELL
#
# Requires:
# - build_support_system_prompt(lab, target_idx)  ✅ (from Option 2 cell)
# - SUPPORT_LABELS (includes "None")
# - client initialized (genai.Client(...))
# - TEXT_COL already set in your pipeline
# ======================================================

from typing import Dict, Any, List
import time, json, re
import pandas as pd
from google.genai import types

MODEL_ID = "gemini-2.0-flash"

# ------------------------------------------------------
# Gemini Safety Settings (lowest blocking)
# ------------------------------------------------------
GEMINI_SAFETY_SETTINGS = [
    types.SafetySetting(
        category=types.HarmCategory.HARM_CATEGORY_HATE_SPEECH,
        threshold=types.HarmBlockThreshold.BLOCK_NONE,
    ),
    types.SafetySetting(
        category=types.HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT,
        threshold=types.HarmBlockThreshold.BLOCK_NONE,
    ),
    types.SafetySetting(
        category=types.HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT,
        threshold=types.HarmBlockThreshold.BLOCK_NONE,
    ),
    types.SafetySetting(
        category=types.HarmCategory.HARM_CATEGORY_HARASSMENT,
        threshold=types.HarmBlockThreshold.BLOCK_NONE,
    ),
]

GEMINI_CONFIG = types.GenerateContentConfig(
    temperature=0.0,
    top_p=0.95,
    max_output_tokens=450,  # support JSON is small; keep this tighter
    safety_settings=GEMINI_SAFETY_SETTINGS,
)

# ------------------------------------------------------
# JSON Loader (robust)
# ------------------------------------------------------
def safe_json_load(txt: str) -> Dict[str, Any]:
    if not txt:
        return {}
    t = str(txt).strip()
    if not t:
        return {}

    # strip fences
    t = re.sub(r"^```json\s*", "", t, flags=re.IGNORECASE).strip()
    t = re.sub(r"^```\s*", "", t).strip()
    t = re.sub(r"\s*```$", "", t).strip()

    # direct parse
    try:
        obj = json.loads(t)
        return obj if isinstance(obj, dict) else {}
    except Exception:
        pass

    # extract last {...}
    matches = re.findall(r"\{.*\}", t, flags=re.DOTALL)
    if not matches:
        return {}
    blob = matches[-1].strip()

    # repairs
    blob = re.sub(r",\s*}", "}", blob)
    blob = re.sub(r",\s*]", "]", blob)
    blob = re.sub(r"\bTrue\b", "true", blob)
    blob = re.sub(r"\bFalse\b", "false", blob)
    blob = re.sub(r"\bNone\b", "null", blob)

    try:
        out = json.loads(blob)
        return out if isinstance(out, dict) else {}
    except Exception:
        return {}

# ------------------------------------------------------
# Detect blocked-ish responses (Gemini sometimes returns empty)
# ------------------------------------------------------
def _is_blocked_like(resp) -> bool:
    txt = (getattr(resp, "text", "") or "").strip()
    if txt:
        return False
    # if no text, treat as blocked/failure
    return True

# ------------------------------------------------------
# Gemini Call
# ------------------------------------------------------
def gemini_prompt_to_json(
    system_prompt: str,
    comment_text: str,
    max_retries: int = 3,
    sleep_s: float = 1.2,
) -> Dict[str, Any]:

    if comment_text is None or (isinstance(comment_text, float) and pd.isna(comment_text)):
        comment_text = ""
    comment_text = str(comment_text)

    base_prompt = (
        f"{system_prompt}\n\n"
        f"COMMENT:\n{comment_text}\n\n"
        "FINAL INSTRUCTIONS:\n"
        "Output ONLY a single JSON object (no markdown, no extra text).\n"
        "Use lowercase true/false.\n"
        'Include exactly these keys: "reasoning", "present", "evidence".\n'
    )

    last_err = None
    last_raw = ""

    for attempt in range(1, max_retries + 1):
        try:
            resp = client.models.generate_content(
                model=MODEL_ID,
                contents=base_prompt,
                config=GEMINI_CONFIG,
            )

            if _is_blocked_like(resp):
                last_err = ValueError("Blocked/empty Gemini output")
                time.sleep(sleep_s)
                continue

            txt = (getattr(resp, "text", "") or "").strip()
            last_raw = txt

            parsed = safe_json_load(txt)
            if not parsed:
                last_err = ValueError(f"JSON parse failed: {txt[:220]}")
                time.sleep(sleep_s)
                continue

            # normalize present
            if "present" in parsed and not isinstance(parsed["present"], bool):
                p = str(parsed["present"]).strip().lower()
                parsed["present"] = True if p == "true" else False

            # validate keys
            if ("present" not in parsed) or ("evidence" not in parsed) or ("reasoning" not in parsed):
                last_err = ValueError(f"Missing required keys: {parsed}")
                time.sleep(sleep_s)
                continue

            # enforce evidence empty if absent
            if not bool(parsed["present"]):
                parsed["evidence"] = ""

            parsed["evidence"] = str(parsed.get("evidence", "") or "").strip()
            parsed["reasoning"] = str(parsed.get("reasoning", "") or "").strip()

            return parsed

        except Exception as e:
            last_err = e
            time.sleep(sleep_s)

    print("❌ Gemini call failed:", last_err)
    if last_raw:
        print("   Last raw snippet:", last_raw[:220])

    return {"_blocked": True, "present": False, "evidence": "", "reasoning": "blocked_or_failed"}

# ======================================================
# ✅ SUPPORT RUNNER (do NOT ask Gemini about "None")
# ======================================================
def run_support(comment_text: str, target_idx: int) -> Dict[str, Any]:

    # split real labels vs None
    REAL_LABELS: List[str] = [l for l in SUPPORT_LABELS if l != "None"]

    final_output = {
        "label_presence": {lab: False for lab in SUPPORT_LABELS},
        "labels": [],
        "evidence": {},
        "explanation": {},
        "_blocked_any": False,
        "_target_idx": int(target_idx),
    }

    if comment_text is None or (isinstance(comment_text, float) and pd.isna(comment_text)):
        comment_text = ""
    comment_text = str(comment_text)

    for lab in REAL_LABELS:
        # ✅ Option 2 signature:
        # system_prompt = build_support_system_prompt(lab=lab, target_idx=int(target_idx))
        system_prompt = cached_system_prompt(lab, int(target_idx))

        res = gemini_prompt_to_json(system_prompt, comment_text)

        if res.get("_blocked"):
            final_output["_blocked_any"] = True

        is_present = bool(res.get("present", False))
        final_output["label_presence"][lab] = is_present
        final_output["explanation"][lab] = res.get("reasoning", "")
        final_output["evidence"][lab] = res.get("evidence", "")

        if is_present:
            final_output["labels"].append(lab)

    # --------------------------------------------------
    # Auto-assign None if no REAL support labels detected
    # --------------------------------------------------
    if len(final_output["labels"]) == 0:
        final_output["labels"] = ["None"]
        final_output["label_presence"]["None"] = True
        final_output["evidence"]["None"] = ""
        final_output["explanation"]["None"] = "No support category matched."
    else:
        final_output["label_presence"]["None"] = False

    return final_output


def predict_support(comment_text: str, target_idx: int) -> Dict[str, Any]:
    try:
        return run_support(comment_text, target_idx)
    except Exception as e:
        print("❌ Fatal Prediction Error:", e)
        lp = {k: False for k in SUPPORT_LABELS}
        lp["None"] = True
        return {
            "label_presence": lp,
            "labels": ["None"],
            "evidence": {},
            "explanation": {"error": str(e)},
            "_blocked_any": True,
            "_target_idx": int(target_idx),
        }

In [ ]:
# ==========================================
# CELL 8 — Support Batch Predict (Parallel + Checkpoint) ✅ COPY-PASTE
# ==========================================

import os
import json
import pandas as pd
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

# --------------------------------------------------
# Load input
# --------------------------------------------------
df_sup = pd.read_csv(SUPPORT_INPUT_PATH)

if TEXT_COL not in df_sup.columns:
    raise ValueError(
        f"Missing TEXT_COL='{TEXT_COL}' in input CSV. Found: {list(df_sup.columns)}"
    )

if ID_COL not in df_sup.columns:
    print(f"⚠️ Warning: ID_COL='{ID_COL}' not found. Preview will skip ID.")

# IMPORTANT: align indices with SIM_MAP
df_sup = df_sup.reset_index(drop=True)

print(f"Total comments to process: {len(df_sup)}")

# --------------------------------------------------
# Checkpoint setup (auto-resume)
# --------------------------------------------------
CHECKPOINT_PATH = "/content/support_checkpoint.jsonl"

done = {}

if os.path.exists(CHECKPOINT_PATH):
    print("🔄 Loading checkpoint...")
    with open(CHECKPOINT_PATH, "r") as f:
        for line in f:
            obj = json.loads(line)
            done[int(obj["_target_idx"])] = obj

print("✅ Already completed:", len(done))

# --------------------------------------------------
# Worker function
# --------------------------------------------------
def process_one(i: int):
    """Runs prediction for one comment (7 Gemini calls inside)."""

    # Skip if already completed
    if i in done:
        return i, done[i]

    raw = df_sup.loc[i, TEXT_COL]
    text = "" if pd.isna(raw) else str(raw)

    pred = predict_support(text, target_idx=i)
    return i, pred


# --------------------------------------------------
# Parallel execution
# --------------------------------------------------
# Start conservative. Increase slowly if stable.
MAX_WORKERS = 8   # try 4 first → then 6 → then 8

results = [None] * len(df_sup)

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:

    futures = [executor.submit(process_one, i) for i in range(len(df_sup))]

    for fut in tqdm(
        as_completed(futures),
        total=len(futures),
        desc="⚡ Predicting Support (parallel)"
    ):

        i, pred = fut.result()
        results[i] = pred

        # ---- checkpoint write immediately ----
        if i not in done:
            with open(CHECKPOINT_PATH, "a") as f:
                f.write(json.dumps(pred, ensure_ascii=False) + "\n")
            done[i] = pred

# --------------------------------------------------
# Save results (same format as original pipeline)
# --------------------------------------------------
df_sup["support_json"] = [
    json.dumps(r, ensure_ascii=False) if r else ""
    for r in results
]

df_sup["support_labels"] = [
    ", ".join(r.get("labels", [])) if isinstance(r, dict) else ""
    for r in results
]

df_sup.to_csv(SUPPORT_OUTPUT_PATH, index=False)

print("\n✅ Saved:", SUPPORT_OUTPUT_PATH)

# --------------------------------------------------
# Preview
# --------------------------------------------------
preview_cols = ["support_labels", "support_json"]
if ID_COL in df_sup.columns:
    preview_cols = [ID_COL] + preview_cols

df_sup[preview_cols].head()

Total comments to process: 1175
✅ Already completed: 0


⚡ Predicting Support (parallel):   0%|          | 0/1175 [00:00<?, ?it/s]

❌ Gemini call failed: Blocked/empty Gemini output
❌ Gemini call failed: JSON parse failed: ```json
{
  "reasoning": "The comment provides a link to a resource about sexual abuse and encourages the recipient to read it. This constitutes providing information to help the recipient understand their situation, thu
   Last raw snippet: ```json
{
  "reasoning": "The comment provides a link to a resource about sexual abuse and encourages the recipient to read it. This constitutes providing information to help the recipient understand their situation, thu
❌ Gemini call failed: Blocked/empty Gemini output

✅ Saved: /content/no_stigma_support_predictions.csv


,comment_id,support_labels,support_json
0,h539sfg,"Information Support, Emotional Support, Esteem...","{""label_presence"": {""Information Support"": tru..."
1,h54ufu8,"Information Support, Emotional Support, Esteem...","{""label_presence"": {""Information Support"": tru..."
2,h54zdn2,"Information Support, Emotional Support, Esteem...","{""label_presence"": {""Information Support"": tru..."
3,h2xdoq5,Tangible Assistance,"{""label_presence"": {""Information Support"": fal..."
4,h2xd7ht,None,"{""label_presence"": {""Information Support"": fal..."


In [ ]:
# ==========================================
# CELL — Expand SUPPORT JSON into Clean Columns ✅ COPY-PASTE (FIXED)
# Fixes:
# - Normalizes label_presence to ALWAYS contain all expected keys
# - Enforces None exclusivity (None true only if no other labels true)
# - Computes support_covered robustly (not blocked + label_presence exists)
# - Expands *_present / *_evidence / *_reasoning cleanly
# ==========================================
print("\nFinalizing data structure...")

import json
import re

# Expand these labels into columns (canonical set)
labels_to_expand = [
    "Information Support",
    "Emotional Support",
    "Esteem Support",
    "Tangible Assistance",
    "Group Interaction",
    "None",   # derived label
]

# ------------------------------------------------------
# Helper: parse support_json (string -> dict)
# ------------------------------------------------------
def _parse_support_json(x):
    if isinstance(x, dict):
        return x
    if isinstance(x, str) and x.strip():
        try:
            return json.loads(x)
        except Exception:
            return {}
    return {}

# ✅ df_sup must exist and contain "support_json"
df_sup["support_json_obj"] = df_sup["support_json"].apply(_parse_support_json)

# ------------------------------------------------------
# Normalize: label_presence keys + None exclusivity
# ------------------------------------------------------
def _normalize_support_obj(obj: Any) -> Dict[str, Any]:
    if not isinstance(obj, dict):
        obj = {}

    # Ensure core keys exist
    if "label_presence" not in obj or not isinstance(obj.get("label_presence"), dict):
        obj["label_presence"] = {}
    if "labels" not in obj or not isinstance(obj.get("labels"), list):
        obj["labels"] = []
    if "evidence" not in obj or not isinstance(obj.get("evidence"), dict):
        obj["evidence"] = {}
    if "explanation" not in obj or not isinstance(obj.get("explanation"), dict):
        obj["explanation"] = {}

    lp = obj["label_presence"]

    # Ensure ALL expected label keys exist in label_presence
    for lab in labels_to_expand:
        if lab not in lp:
            lp[lab] = False
        else:
            # normalize truthy values to strict bool
            lp[lab] = True if lp[lab] is True else False

    # Enforce None exclusivity:
    any_real = any(lp.get(lab, False) for lab in labels_to_expand if lab != "None")
    lp["None"] = False if any_real else True

    # Make labels list consistent with label_presence (source of truth)
    obj["labels"] = [lab for lab in labels_to_expand if lp.get(lab, False)]

    # Evidence should only exist for labels that are present (optional cleanup)
    obj["evidence"] = {
        k: str(v).strip()
        for k, v in (obj.get("evidence", {}) or {}).items()
        if k in obj["labels"] and str(v).strip()
    }

    obj["label_presence"] = lp
    return obj

df_sup["support_json_obj"] = df_sup["support_json_obj"].apply(_normalize_support_obj)

# ------------------------------------------------------
# 0) Blocked / Covered markers (robust)
# Covered = not blocked + has label_presence dict
# ------------------------------------------------------
df_sup["support_blocked_any"] = df_sup["support_json_obj"].apply(
    lambda x: bool(x.get("_blocked_any", False)) if isinstance(x, dict) else True
)

df_sup["support_covered"] = df_sup["support_json_obj"].apply(
    lambda x: (
        isinstance(x, dict)
        and (not bool(x.get("_blocked_any", False)))
        and isinstance(x.get("label_presence", None), dict)
    )
)

# ------------------------------------------------------
# 1) Expand per-label fields
# ------------------------------------------------------
for label in labels_to_expand:
    safe_col = re.sub(r"[^A-Za-z0-9]+", "_", label).strip("_")  # safe column names

    df_sup[f"{safe_col}_present"] = df_sup["support_json_obj"].apply(
        lambda x: x.get("label_presence", {}).get(label, False) if isinstance(x, dict) else False
    )

    df_sup[f"{safe_col}_evidence"] = df_sup["support_json_obj"].apply(
        lambda x: x.get("evidence", {}).get(label, "") if isinstance(x, dict) else ""
    )

    df_sup[f"{safe_col}_reasoning"] = df_sup["support_json_obj"].apply(
        lambda x: x.get("explanation", {}).get(label, "") if isinstance(x, dict) else ""
    )

# ------------------------------------------------------
# 2) Summary labels string (comma-separated)
# ------------------------------------------------------
df_sup["support_labels"] = df_sup["support_json_obj"].apply(
    lambda x: ", ".join(x.get("labels", [])) if isinstance(x, dict) else ""
)

# ------------------------------------------------------
# 3) Save
# ------------------------------------------------------
df_sup.to_csv(SUPPORT_OUTPUT_PATH, index=False)
print(f"\nSUCCESS! Results saved to: {SUPPORT_OUTPUT_PATH}")

# ------------------------------------------------------
# 4) Useful run stats
# ------------------------------------------------------
total_with_any_label = df_sup["support_labels"].apply(lambda x: len(str(x).strip()) > 0).sum()
print(f"Total items with any label string: {total_with_any_label}")

covered_n = int(df_sup["support_covered"].sum())
blocked_n = int(df_sup["support_blocked_any"].sum())
print(f"Coverage (unblocked & valid): {covered_n}/{len(df_sup)} = {covered_n/len(df_sup):.1%}")
print(f"Blocked/failed (any): {blocked_n}/{len(df_sup)} = {blocked_n/len(df_sup):.1%}")


Finalizing data structure...

SUCCESS! Results saved to: /content/support_predictions.csv
Total items with any label string: 500
Coverage (unblocked & valid): 498/500 = 99.6%
Blocked/failed (any): 2/500 = 0.4%


In [ ]:
# =========================
# CELL A — Install/Imports
# =========================
import pandas as pd
import numpy as np

from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report
)

# Paths
L3_PRED_PATH = "/content/support_predictions.csv"
L3_GT_PATH   = "/content/Sexual Violence sampled data - 500 random sampling.csv"


# Columns
ID_COL = "comment_id"
GT_COL = "Tag"
PRED_COL = "support_labels"


# Canonical label order (must match both GT + preds)
LABELS = ["Information Support",
    "Emotional Support",
    "Esteem Support",
    "Tangible Assistance",
    "Group Interaction",]

# --- quick sanity checks (helps catch silent column mismatch) ---
df_pred = pd.read_csv(L3_PRED_PATH)
df_gt   = pd.read_csv(L3_GT_PATH)

missing = [c for c in [ID_COL, PRED_COL] if c not in df_pred.columns]
if missing:
    raise ValueError(f"Pred file missing columns: {missing}. Found: {list(df_pred.columns)}")

if ID_COL not in df_gt.columns or GT_COL not in df_gt.columns:
    raise ValueError(f"GT file must have columns '{ID_COL}' and '{GT_COL}'. Found: {list(df_gt.columns)}")

print("✅ Loaded pred rows:", len(df_pred), "| gt rows:", len(df_gt))
print("✅ Pred columns ok.")
print("✅ GT columns ok.")

print("\nPred label examples:")
print(df_pred[PRED_COL].dropna().head(5).tolist())

✅ Loaded pred rows: 500 | gt rows: 500
✅ Pred columns ok.
✅ GT columns ok.

Pred label examples:
['Information Support, Emotional Support, Esteem Support', 'Information Support, Emotional Support, Esteem Support', 'Information Support, Emotional Support, Tangible Assistance, Group Interaction', 'Information Support', 'Information Support, Emotional Support, Esteem Support']


In [ ]:
# ==========================================
# CELL B — Helpers: Robust SUPPORT Label Parsing
# ==========================================

import numpy as np

# Canonical support labels (include None because it's part of final output)
ALL_SUPPORT_LABELS = SUPPORT_LABELS + ["None"]

# Case-insensitive lookup map
LABEL_MAP = {lab.lower(): lab for lab in ALL_SUPPORT_LABELS}

# ---- ADD THIS ----
ALIASES = {
    "group interaction": "Group Interaction",
}

# Merge aliases into label map
for k, v in ALIASES.items():
    LABEL_MAP[k] = v

def parse_labels_cell(x):
    """
    Handles:
    - Lists (['Emotional Support', 'Information Support'])
    - Strings ("Emotional Support, Information Support")
    - Strings with semicolons
    - NaNs / empty
    Normalizes case-insensitively to canonical support labels.
    """

    # 1) list-like
    if isinstance(x, list):
        out = []
        seen = set()
        for item in x:
            if not isinstance(item, str):
                continue
            key = item.strip().lower()
            if key in LABEL_MAP and LABEL_MAP[key] not in seen:
                out.append(LABEL_MAP[key])
                seen.add(LABEL_MAP[key])
        return out

    # 2) NaN / None
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return []

    # 3) string
    s = str(x).strip()
    if not s or s.lower() == "nan":
        return []

    # Allow ; or , separators
    parts = [p.strip() for p in s.replace(";", ",").split(",") if p.strip()]

    out = []
    seen = set()

    for p in parts:
        key = p.lower()
        if key in LABEL_MAP:
            lab = LABEL_MAP[key]
            if lab not in seen:
                out.append(lab)
                seen.add(lab)

    return out


def set_str(lbls):
    """Sort + join labels for exact-match comparisons."""
    if not lbls:
        return ""
    return ",".join(sorted(lbls))

In [ ]:
import pandas as pd
import numpy as np
import re

# ==========================================
# CELL — Load + Merge + Evaluate SUPPORT (Coverage-aware)
# ==========================================

df_pred = pd.read_csv(L3_PRED_PATH)   # predictions file (expanded)
df_gt   = pd.read_csv(L3_GT_PATH)     # ground truth file


# Canonical label list (matches your annotation scheme)
SUPPORT_EVAL_LABELS = [
    "Information Support",
    "Emotional Support",
    "Esteem Support",
    "Tangible Assistance",
    "Group Interaction",
    "None",
]

def _safe_col(label: str) -> str:
    """Must match the sanitization you used when expanding JSON into columns."""
    return re.sub(r"[^A-Za-z0-9]+", "_", label).strip("_")

# -------------------------------
# Helper: robust bool parsing (handles True/False, 1/0, "true"/"false")
# -------------------------------
def _to_bool(v) -> bool:
    if v is None:
        return False
    if isinstance(v, (bool, np.bool_)):
        return bool(v)
    if isinstance(v, (int, float, np.integer, np.floating)):
        if isinstance(v, float) and np.isnan(v):
            return False
        return v != 0
    if isinstance(v, str):
        s = v.strip().lower()
        if s in {"true", "t", "1", "yes", "y"}:
            return True
        if s in {"false", "f", "0", "no", "n", ""}:
            return False
        return False
    return False

# -------------------------------
# 1) Reconstruct pred_list from *_present columns
#    NOTE: columns are sanitized (underscored), e.g. Information_Support_present
# -------------------------------
def get_labels_from_presence(row):
    active = []
    for lbl in SUPPORT_EVAL_LABELS:
        col = f"{_safe_col(lbl)}_present"
        if col in row and _to_bool(row[col]):
            active.append(lbl)

    # Enforce exclusivity for None:
    # if any real support labels exist, drop None
    if len(active) > 1 and "None" in active:
        active = [x for x in active if x != "None"]

    # if nothing triggered, default to None (safety)
    if not active:
        active = ["None"]

    return active

df_pred["pred_list_reconstructed"] = df_pred.apply(get_labels_from_presence, axis=1)

# -------------------------------
# 2) Merge GT + Pred by Comment ID
# -------------------------------
if ID_COL in df_pred.columns and ID_COL in df_gt.columns:
    keep_cols = [ID_COL, "pred_list_reconstructed"]

    if "support_covered" in df_pred.columns:
        keep_cols.append("support_covered")
    if "support_blocked_any" in df_pred.columns:
        keep_cols.append("support_blocked_any")

    merged = df_gt[[ID_COL, GT_COL]].merge(
        df_pred[keep_cols],
        on=ID_COL,
        how="inner"
    )
    print("Merged rows (by Comment ID):", len(merged))
else:
    raise ValueError(f"Missing '{ID_COL}' in pred and/or gt files.")

# -------------------------------
# 3) Parse GT labels + attach predictions
# -------------------------------
merged["gt_list"] = merged[GT_COL].apply(parse_labels_cell)
merged["pred_list"] = merged["pred_list_reconstructed"]

merged["gt_set"]   = merged["gt_list"].apply(lambda x: set(x) if isinstance(x, list) else set())
merged["pred_set"] = merged["pred_list"].apply(lambda x: set(x) if isinstance(x, list) else set())

# Enforce exclusivity for None in GT too (optional but recommended)
merged["gt_set"] = merged["gt_set"].apply(lambda s: (s - {"None"}) if ("None" in s and len(s) > 1) else s)

# -------------------------------
# 4) Coverage filtering (predicted-only)
# -------------------------------
covered = merged.copy()

# if "support_covered" in covered.columns:
#     covered = covered[covered["support_covered"] == True]

# if "support_blocked_any" in covered.columns:
#     covered = covered[covered["support_blocked_any"] == False]

print(f"Rows used for metrics: {len(covered)}/{len(merged)} = {len(covered)/len(merged):.1%}")

# -------------------------------
# 5) Exact-match Accuracy (on covered subset)
# -------------------------------
covered["exact_match"] = (covered["gt_set"] == covered["pred_set"])

print("-" * 30)
print(f"Exact-match Accuracy (COVERED only): {covered['exact_match'].mean():.4f}")
print("-" * 30)

mismatches = covered[covered["exact_match"] == False]
if len(mismatches) > 0:
    ex = mismatches.iloc[0]
    print(f"Sample mismatch (Comment ID {ex[ID_COL]}):")
    print(f"  GT:   {ex['gt_set']}")
    print(f"  PRED: {ex['pred_set']}")
else:
    print("✅ No mismatches found (perfect exact match on covered rows).")
for c in ["support_covered","support_blocked_any","l3_covered","l3_blocked_any"]:
    if c in df_pred.columns:
        print("\n", c)
        print(df_pred[c].value_counts(dropna=False).head(10))

Merged rows (by Comment ID): 500
Rows used for metrics: 500/500 = 100.0%
------------------------------
Exact-match Accuracy (COVERED only): 0.4460
------------------------------
Sample mismatch (Comment ID gnxyazs):
  GT:   {'Information Support', 'Emotional Support'}
  PRED: {'Esteem Support', 'Emotional Support', 'Information Support'}

 support_covered
support_covered
True     498
False      2
Name: count, dtype: int64

 support_blocked_any
support_blocked_any
False    498
True       2
Name: count, dtype: int64


In [ ]:
df_gt

,post_id,post_title,post_url,post_body,comment_id,author,comment_text,Tag,l2_label,l3_labels
0,7qhyct,Last year. I was 29. I went on a date with a g...,https://www.reddit.com/r/meToo/comments/7qhyct...,After forcing my hand to spank her ass. She th...,dsrqe0q,FuckOffMightBe2Kind,What did i just read....,none,Stigma,Internalized
1,lmsnrm,Am i being dramatic?,https://www.reddit.com/r/sexualassault/comment...,I’m 17 and my boyfriend at the time was 18. He...,gnxyazs,HooDooVooDoo666,Yes they are. When you said it hurt he Should ...,"emotional support, information support",Stigma,Internalized
2,kt2e7m,i think my cousin tried to r@pe me when i was 9?,https://www.reddit.com/r/sexualassault/comment...,(my memory is so blurry of this event so i apo...,gik5xo7,HereComesNancyDrew,"100% was this assault. You were a child, and h...","information support, emotional support",Stigma,NaN
3,nyc6ec,I just want to feel like me again,https://www.reddit.com/r/sexualassault/comment...,I haven't posted about my story because its ha...,h1jvp91,AhsokaLivesMatter,I am so sorry to hear about this! And believe ...,"emotional support, tangible assistance, esteem...",Stigma,Internalized
4,n41sgh,I don't know what to do about this guy? Report...,https://www.reddit.com/r/sexualassault/comment...,So this guy has basically stalked many many ma...,h0menfc,surreaLies,Protect yourself. Dudes like this hide behind ...,information support,Stigma,"Experienced, Structural"
...,...,...,...,...,...,...,...,...,...,...
495,jkss23,"Please help, was this sexual assault?",https://www.reddit.com/r/sexualassault/comment...,I’ve posted several times with no response.\n ...,gakxzfj,hollow_bastien,Yes.,none,Stigma,Internalized
496,jquhxt,i didn’t say no while it happened.,https://www.reddit.com/r/sexualassault/comment...,"i’m really just looking for answers, a few mon...",gbprzo9,worriedplantparent,Hey I'm so sorry that happened to you. It's ve...,"emotional support, esteem support, information...",Stigma,"Internalized, Anticipated"
497,n3kamo,Do you think I was drugged? This is breaking me.,https://www.reddit.com/r/sexualassault/comment...,Thank you for choosing to read my post\n \n ...,gwqhj37,TuffyManzer,It sounds like she had laced it with something...,information support,Stigma,"Internalized, Anticipated"
498,bjlafw,Advice? I think I might be dealing with a groo...,https://www.reddit.com/r/meToo/comments/bjlafw...,"First time poster in here, but felt like this ...",ema2vor,fingerkuffs23,Ugh. This makes me so mad. And I know that the...,"information support, emotional support, tangib...",Stigma,Internalized


In [ ]:
df_pred

,post_id,post_title,post_url,post_body,comment_id,author,comment_text,l2_label,l3_labels,support_json,...,Tangible_Assistance_present,Tangible_Assistance_evidence,Tangible_Assistance_reasoning,Group_Interaction_present,Group_Interaction_evidence,Group_Interaction_reasoning,None_present,None_evidence,None_reasoning,pred_list_reconstructed
0,7qhyct,Last year. I was 29. I went on a date with a g...,https://www.reddit.com/r/meToo/comments/7qhyct...,After forcing my hand to spank her ass. She th...,dsrqe0q,FuckOffMightBe2Kind,What did i just read....,Stigma,Internalized,"{""label_presence"": {""Information Support"": fal...",...,False,NaN,The comment does not offer direct personal ass...,False,NaN,The comment expresses shock/disbelief but does...,True,NaN,No support category matched.,[None]
1,lmsnrm,Am i being dramatic?,https://www.reddit.com/r/sexualassault/comment...,I’m 17 and my boyfriend at the time was 18. He...,gnxyazs,HooDooVooDoo666,Yes they are. When you said it hurt he Should ...,Stigma,Internalized,"{""label_presence"": {""Information Support"": tru...",...,False,NaN,The comment does not offer direct personal ass...,False,NaN,The comment expresses sympathy but lacks expli...,False,NaN,NaN,"[Information Support, Emotional Support, Estee..."
2,kt2e7m,i think my cousin tried to r@pe me when i was 9?,https://www.reddit.com/r/sexualassault/comment...,(my memory is so blurry of this event so i apo...,gik5xo7,HereComesNancyDrew,"100% was this assault. You were a child, and h...",Stigma,NaN,"{""label_presence"": {""Information Support"": tru...",...,False,NaN,The comment does not offer direct personal ass...,False,NaN,The comment expresses sympathy and acknowledge...,False,NaN,NaN,"[Information Support, Emotional Support, Estee..."
3,nyc6ec,I just want to feel like me again,https://www.reddit.com/r/sexualassault/comment...,I haven't posted about my story because its ha...,h1jvp91,AhsokaLivesMatter,I am so sorry to hear about this! And believe ...,Stigma,Internalized,"{""label_presence"": {""Information Support"": tru...",...,True,DM me if you’d like to talk more about that.,The comment explicitly connects the recipient ...,True,I know this feeling all too well.,The comment includes both an expression of emp...,False,NaN,NaN,"[Information Support, Emotional Support, Tangi..."
4,n41sgh,I don't know what to do about this guy? Report...,https://www.reddit.com/r/sexualassault/comment...,So this guy has basically stalked many many ma...,h0menfc,surreaLies,Protect yourself. Dudes like this hide behind ...,Stigma,"Experienced, Structural","{""label_presence"": {""Information Support"": tru...",...,False,NaN,The comment does not offer direct personal ass...,False,NaN,The comment primarily offers advice and sugges...,False,NaN,NaN,[Information Support]
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,jkss23,"Please help, was this sexual assault?",https://www.reddit.com/r/sexualassault/comment...,I’ve posted several times with no response.\n ...,gakxzfj,hollow_bastien,Yes.,Stigma,Internalized,"{""label_presence"": {""Information Support"": fal...",...,False,NaN,"The comment ""Yes."" does not contain any offer ...",False,NaN,"The comment ""Yes."" does not contain any explic...",True,NaN,No support category matched.,[None]
496,jquhxt,i didn’t say no while it happened.,https://www.reddit.com/r/sexualassault/comment...,"i’m really just looking for answers, a few mon...",gbprzo9,worriedplantparent,Hey I'm so sorry that happened to you. It's ve...,Stigma,"Internalized, Anticipated","{""label_presence"": {""Information Support"": tru...",...,False,NaN,The comment provides emotional support and rea...,False,NaN,The comment expresses sympathy and normalizes ...,False,NaN,NaN,"[Information Support, Emotional Support, Estee..."
497,n3kamo,Do you think I was drugged? This is breaking me.,https://www.reddit.com/r/sexualassault/comment...,Thank you for choosing to read my post\n \n ...,gwqhj37,TuffyManzer,It s

In [ ]:
print(merged['gt_set'].iloc[0], merged['pred_set'].iloc[0])

{'None'} {'None'}


In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer

FINE_LABELS = LABELS
mlb = MultiLabelBinarizer(classes=FINE_LABELS)

LABEL_MAP = {lab.lower(): lab for lab in FINE_LABELS}

def split_labels(x):
    """
    Accepts:
    - list (['Experienced', 'Anticipated'])
    - string ("Experienced, Anticipated")
    - NaN / empty
    Returns canonical labels aligned with FINE_LABELS.
    """
    # list
    if isinstance(x, list):
        out = []
        seen = set()
        for item in x:
            if not isinstance(item, str):
                continue
            key = item.strip().lower()
            if key in LABEL_MAP:
                lab = LABEL_MAP[key]
                if lab not in seen:
                    out.append(lab)
                    seen.add(lab)
        return out

    # NaN / None
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return []

    # string
    s = str(x).strip()
    if not s or s.lower() == "nan":
        return []

    parts = [p.strip() for p in s.replace(";", ",").split(",") if p.strip()]

    out = []
    seen = set()
    for p in parts:
        key = p.lower()
        if key in LABEL_MAP:
            lab = LABEL_MAP[key]
            if lab not in seen:
                out.append(lab)
                seen.add(lab)

    return out

In [ ]:
from sklearn.metrics import classification_report
import pandas as pd

# ✅ Use COVERED subset (predicted-only). If you don't have it, fallback to merged.
df_eval = covered if "covered" in globals() else merged

# --- safety: ensure lists ---
df_eval["gt_list_clean"] = df_eval["gt_list"].apply(parse_labels_cell)
df_eval["pred_list_clean"] = df_eval["pred_list"].apply(parse_labels_cell)

# --- lock class order explicitly ---
mlb = MultiLabelBinarizer(classes=FINE_LABELS)
mlb.fit([[]])  # ensures internal class order is exactly FINE_LABELS

# 1) Binary matrices
y_true = mlb.transform(df_eval["gt_list_clean"])
y_pred = mlb.transform(df_eval["pred_list_clean"])

# 2) Report (text)
report = classification_report(
    y_true,
    y_pred,
    target_names=FINE_LABELS,
    zero_division=0
)

print("Classification Report for Level 3 Stigma Labels (COVERED only):")
print("-" * 60)
print(report)
print("-" * 60)

# 3) Report (DataFrame)
report_dict = classification_report(
    y_true, y_pred,
    target_names=FINE_LABELS,
    output_dict=True,
    zero_division=0
)
df_report = pd.DataFrame(report_dict).transpose()

df_report

Classification Report for Level 3 Stigma Labels (COVERED only):
------------------------------------------------------------
                     precision    recall  f1-score   support

Information Support       0.95      0.86      0.90       410
  Emotional Support       0.80      0.88      0.84       221
     Esteem Support       0.74      0.82      0.78       232
Tangible Assistance       0.72      0.78      0.75        55
  Group Interaction       0.72      0.73      0.73       134

          micro avg       0.82      0.83      0.83      1052
          macro avg       0.79      0.81      0.80      1052
       weighted avg       0.83      0.83      0.83      1052
        samples avg       0.77      0.78      0.75      1052

------------------------------------------------------------


,precision,recall,f1-score,support
Information Support,0.953804,0.856098,0.902314,410.0
Emotional Support,0.801653,0.877828,0.838013,221.0
Esteem Support,0.736434,0.818966,0.775510,232.0
Tangible Assistance,0.716667,0.781818,0.747826,55.0
Group Interaction,0.720588,0.731343,0.725926,134.0
micro avg,0.823308,0.832700,0.827977,1052.0
macro avg,0.785829,0.813211,0.797918,1052.0
weighted avg,0.831800,0.832700,0.830297,1052.0
samples avg,0.765167,0.776033,0.747554,1052.0


In [ ]:
from sklearn.metrics import precision_recall_fscore_support

p_micro, r_micro, f1_micro, _ = precision_recall_fscore_support(
    y_true,
    y_pred,
    average="micro",
    labels=range(len(FINE_LABELS)),
    zero_division=0
)

print("LEVEL 3 (Micro)")
print("Precision:", round(p_micro, 4))
print("Recall:   ", round(r_micro, 4))
print("F1:       ", round(f1_micro, 4))


LEVEL 3 (Micro)
Precision: 0.8233
Recall:    0.8327
F1:        0.828


In [ ]:
p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(
    y_true,
    y_pred,
    average="macro",
    labels=range(len(FINE_LABELS)),
    zero_division=0
)

print("\nLEVEL 3 (Macro)")
print("Precision:", round(p_macro, 4))
print("Recall:   ", round(r_macro, 4))
print("F1:       ", round(f1_macro, 4))


LEVEL 3 (Macro)
Precision: 0.7858
Recall:    0.8132
F1:        0.7979


In [ ]:
p_lbl, r_lbl, f1_lbl, support = precision_recall_fscore_support(
    y_true, y_pred, average=None, zero_division=0
)

for i, label in enumerate(FINE_LABELS):
    print(f"{label:12s} | P: {p_lbl[i]:.3f} | R: {r_lbl[i]:.3f} | F1: {f1_lbl[i]:.3f} | Support: {support[i]}")

Information Support | P: 0.954 | R: 0.856 | F1: 0.902 | Support: 410
Emotional Support | P: 0.802 | R: 0.878 | F1: 0.838 | Support: 221
Esteem Support | P: 0.736 | R: 0.819 | F1: 0.776 | Support: 232
Tangible Assistance | P: 0.717 | R: 0.782 | F1: 0.748 | Support: 55
Group Interaction | P: 0.721 | R: 0.731 | F1: 0.726 | Support: 134


In [ ]:
df_gt[GT_COL].astype(str).str.contains("group interaction", case=False, na=False).sum()

np.int64(82)

In [ ]:
from sklearn.metrics import precision_recall_fscore_support, classification_report

# --------------------------------------
# 0) Basic counts
# --------------------------------------
total_posts = len(merged)

if "l3_covered" in merged.columns:
    covered_df = merged[merged["l3_covered"] == True].copy()
    blocked_df = merged[merged["l3_covered"] == False].copy()
else:
    covered_df = merged.copy()
    blocked_df = merged.iloc[0:0]

print("\nDATASET COVERAGE SUMMARY")
print("-" * 50)
print(f"Total posts in GT:          {total_posts}")
print(f"Posts evaluated (covered): {len(covered_df)}")
print(f"Posts ignored (blocked):   {len(blocked_df)}")
print(f"Coverage rate:             {len(covered_df)/total_posts:.1%}")
print("-" * 50)


# --------------------------------------
# 1) METRICS — COVERED POSTS ONLY
# --------------------------------------
y_true_cov = mlb.fit_transform(covered_df["gt_list"])
y_pred_cov = mlb.transform(covered_df["pred_list"])

p_micro, r_micro, f1_micro, _ = precision_recall_fscore_support(
    y_true_cov, y_pred_cov, average="micro",
    labels=range(len(FINE_LABELS)), zero_division=0
)

p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(
    y_true_cov, y_pred_cov, average="macro",
    labels=range(len(FINE_LABELS)), zero_division=0
)

print("\nLEVEL 3 METRICS — COVERED POSTS ONLY")
print("-" * 50)
print(f"Micro  P/R/F1: {p_micro:.3f} / {r_micro:.3f} / {f1_micro:.3f}")
print(f"Macro  P/R/F1: {p_macro:.3f} / {r_macro:.3f} / {f1_macro:.3f}")
print("-" * 50)


# --------------------------------------
# 2) METRICS — INCLUDING BLOCKED POSTS
#    (blocked = predict nothing)
# --------------------------------------
# For blocked rows: pred_list = []
blocked_df["pred_list"] = [[] for _ in range(len(blocked_df))]

all_eval = pd.concat([covered_df, blocked_df], axis=0)

y_true_all = mlb.fit_transform(all_eval["gt_list"])
y_pred_all = mlb.transform(all_eval["pred_list"])

p_micro_all, r_micro_all, f1_micro_all, _ = precision_recall_fscore_support(
    y_true_all, y_pred_all, average="micro",
    labels=range(len(FINE_LABELS)), zero_division=0
)

p_macro_all, r_macro_all, f1_macro_all, _ = precision_recall_fscore_support(
    y_true_all, y_pred_all, average="macro",
    labels=range(len(FINE_LABELS)), zero_division=0
)

print("\nLEVEL 3 METRICS — INCLUDING BLOCKED POSTS")
print("(blocked posts treated as predicting NO labels)")
print("-" * 50)
print(f"Micro  P/R/F1: {p_micro_all:.3f} / {r_micro_all:.3f} / {f1_micro_all:.3f}")
print(f"Macro  P/R/F1: {p_macro_all:.3f} / {r_macro_all:.3f} / {f1_macro_all:.3f}")
print("-" * 50)


# --------------------------------------
# 3) OPTIONAL: Per-label report (covered only)
# --------------------------------------
print("\nPER-LABEL REPORT — COVERED ONLY")
print("-" * 50)
print(classification_report(
    y_true_cov,
    y_pred_cov,
    target_names=FINE_LABELS,
    zero_division=0
))


DATASET COVERAGE SUMMARY
--------------------------------------------------
Total posts in GT:          76
Posts evaluated (covered): 73
Posts ignored (blocked):   3
Coverage rate:             96.1%
--------------------------------------------------

LEVEL 3 METRICS — COVERED POSTS ONLY
--------------------------------------------------
Micro  P/R/F1: 0.743 / 0.873 / 0.803
Macro  P/R/F1: 0.713 / 0.837 / 0.769
--------------------------------------------------

LEVEL 3 METRICS — INCLUDING BLOCKED POSTS
(blocked posts treated as predicting NO labels)
--------------------------------------------------
Micro  P/R/F1: 0.743 / 0.833 / 0.786
Macro  P/R/F1: 0.713 / 0.806 / 0.756
--------------------------------------------------

PER-LABEL REPORT — COVERED ONLY
--------------------------------------------------
              precision    recall  f1-score   support

 Experienced       0.70      0.85      0.77        39
Internalized       0.86      0.94      0.90        52
 Anticipated       0.

In [ ]:
empty_preds = covered[covered["pred_set"].apply(len) == 0]

print(f"Empty predictions (covered): {len(empty_preds)}")
for i, row in empty_preds.iterrows():
    print("-" * 60)
    print("Post ID:", row[ID_COL])
    print("GT labels:", row["gt_set"])

Empty predictions (covered): 5
------------------------------------------------------------
Post ID: 949vnn
GT labels: {'Internalized'}
------------------------------------------------------------
Post ID: mgr4uj
GT labels: {'Structural'}
------------------------------------------------------------
Post ID: od6nku
GT labels: {'Internalized'}
------------------------------------------------------------
Post ID: m41tdc
GT labels: {'Internalized'}
------------------------------------------------------------
Post ID: kjme6y
GT labels: {'Experienced', 'Internalized'}


In [ ]:
print(df_l3.columns.tolist())

['Post ID', 'body', 'l3_json', 'Experienced_present', 'Experienced_evidence', 'Experienced_reasoning', 'Internalized_present', 'Internalized_evidence', 'Internalized_reasoning', 'Anticipated_present', 'Anticipated_evidence', 'Anticipated_reasoning', 'Structural_present', 'Structural_evidence', 'Structural_reasoning', 'l3_labels']


In [ ]:
print(df_human.columns.tolist())

['Post ID', 'body', 'l3_json', 'Experienced_present', 'Experienced_evidence', 'Experienced_reasoning', 'Internalized_present', 'Internalized_evidence', 'Internalized_reasoning', 'Anticipated_present', 'Anticipated_evidence', 'Anticipated_reasoning', 'Structural_present', 'Structural_evidence', 'Structural_reasoning', 'l3_labels']


In [ ]:
# Load your human-coded gold standard
df_human = pd.read_csv("/content/Sexual Violence sampled data - Level_3_annotation.csv")

# Merge it with your AI results
# (Ensure both dataframes have a 'Post ID' column)
df_final = df_l3.merge(df_human[['Post ID', 'Tags']], on='Post ID', how='left')
df_final = df_final.rename(columns={'Tags': 'ground_truth_labels'})

In [ ]:
# 1. Load your human-coded gold standard
df_human = pd.read_csv("/content/Sexual Violence sampled data - Level_3_annotation.csv")

# 2. Merge it with your AI results
# This creates a combined table called 'df_final'
df_final = df_l3.merge(df_human[['Post ID', 'Tags']], on='Post ID', how='left')
df_final = df_final.rename(columns={'Tags': 'ground_truth_labels'})

# 3. Clean the columns in df_final (the merged table)
df_final['gt_clean'] = df_final['ground_truth_labels'].astype(str).str.replace("[", "").str.replace("]", "").str.replace("'", "").str.strip()
df_final['model_clean'] = df_final['l3_labels'].astype(str).str.replace("[", "").str.replace("]", "").str.replace("'", "").str.strip()

# 4. Identify the disagreements
disagreements = df_final[df_final["gt_clean"] != df_final["model_clean"]]

print(f"Total Disagreements: {len(disagreements)}")
print("=" * 50)

# 5. Print the Audit Report
for i, row in disagreements.head(10).iterrows():
    print(f"POST ID: {row['Post ID']}")
    print(f"TEXT: {row['body'][:200]}...")
    print(f"ACTUAL (GT): {row['gt_clean']}")
    print(f"PREDICTED:   {row['model_clean']}")

    # Show Reasoning for the specific categories involved
    print("\nMODEL REASONING:")
    for label in ["Experienced", "Anticipated", "Internalized", "Structural"]:
        if label in str(row['model_clean']) or label in str(row['gt_clean']):
            # Access the specific reasoning column for that label
            reason = row.get(f'{label}_reasoning', 'No reasoning found')
            print(f"-> {label}: {reason}")

    print("-" * 50)

Total Disagreements: 63
POST ID: jzs780
TEXT: I was assaulted while drunk in March in a different country just before lockdown. 
  It’s still quiet raw for me and I am getting trauma Councelling. 
  I still feel as though it was my fault for goin...
ACTUAL (GT): Anticipated, Internalized
PREDICTED:   Experienced, Internalized, Anticipated

MODEL REASONING:
-> Experienced: The boyfriend's statement that the survivor's 'negativity is rubbing off on him' and causing issues in their relationship indicates a negative reaction from someone close to the survivor. This constitutes a stigmatizing social interaction.
-> Anticipated: The survivor anticipates blame from her boyfriend if she discloses the assault ("I feel like he'd blame me"). The Actor is her boyfriend, and the Action/Fear is that he will blame her if she tells him. Therefore, 'present' is true.
-> Internalized: The survivor blames herself for the assault, stating, "I still feel as though it was my fault for going with strangers."